# Unidad III – Machine Learning con Enfoque en Aplicaciones

## Clase 3 – Desarrollo Orientado a Servicios

* **Curso:** IA y Ciencia de Datos (Certificado – 1 año)
* **Enfoque:** Práctico, orientado a producción
* **Tecnologías base:** Python, FastAPI, Pydantic, Scikit-learn

---


### Objetivo general de la clase

Al finalizar la clase, el estudiante será capaz de:

> **Diseñar, evaluar y mejorar una API de Machine Learning preparada para entornos productivos**, aplicando validaciones estrictas, manejo profesional de errores y principios de arquitectura orientada a servicios.

---


## 1. Desarrollo Orientado a Servicios (Service-Oriented Development)

### 1.1 ¿Qué es desarrollo orientado a servicios?

Basado en **Service-Oriented Architecture (SOA)** y **Microservices**.

#### Definición formal

Un **servicio** es una unidad de software que:

* Expone una **interfaz clara**
* Es **independiente**
* Se comunica mediante protocolos estándar (HTTP/REST)
* Tiene una responsabilidad única

---


### 1.2 API como contrato

Una API **no es código**, es un **contrato**.

Define:

* Qué recibe
* Qué devuelve
* Qué errores puede producir

En ML:

* El modelo **no se expone directamente**
* Se expone **un servicio de inferencia**

---


<img src="img/Cómo-funciona-Restful.png" width="1300" alt="Descripción">

### 2. Buenas prácticas para construir APIs robustas

#### 2.1 Principios clave (industria)

| Principio             | Descripción                     |
| --------------------- | ------------------------------- |
| Single Responsibility | Un endpoint = un propósito      |
| Stateless             | No guarda estado entre requests |
| Validación estricta   | Nunca confiar en el cliente     |
| Errores explícitos    | Nunca fallar silenciosamente    |
| Tipado fuerte         | Reduce errores en producción    |

---


### 2.2 Estructura profesional de una API ML

```
app/
 ├── api/
 │   └── routes.py
 ├── service/
 │   └── model_service.py
 ├── pipeline/
 │   └── pipeline.py
 ├── model/
 │   └── pipeline.joblib
 └── main.py
```

✔ Separación clara de responsabilidades
✔ Facilita pruebas y mantenimiento
✔ Escalable

---


### 2.3 Buenas prácticas específicas para APIs de ML

* No entrenar modelos en endpoints
* Cargar el modelo una sola vez (startup)
* Validar **orden y tipo de features**
* Controlar inputs fuera de rango
* No exponer detalles internos del modelo

---


### 3. Validaciones y manejo de errores profesional

* Datos inválidos
* Errores silenciosos
* Predicciones absurdas

---


#### 3.1 Manejo correcto de errores

* Mensaje claro
* Código correcto
* Sin filtrar detalles internos

---


# Enunciado del Proyecto – Módulo 3 (Data Science & AI)

## **Objetivo**

Construir una aplicación de Machine Learning completa y profesional, que entrene un modelo con datos reales y exponga predicciones mediante una API REST con FastAPI.

### **Contexto**

Trabajarás con un dataset real de consumo energético doméstico para predecir el consumo de electrodomésticos (Wh). El enfoque será de regresión y deberás aplicar buenas prácticas de ingeniería de software, ML en producción y validaciones estrictas de entrada.

### **Requisitos obligatorios**

#### **Dataset real desde internet**
* Descargar desde URL pública.
* Cachear localmente.
#### **Pipeline profesional (Scikit-learn)**
* SimpleImputer + StandardScaler + ColumnTransformer
* Modelo: ElasticNet
#### **Entrenamiento y evaluación**
* Train/test split
* Métricas: MAE, MSE, RMSE, R²
* Guardar el pipeline con joblib
#### **API REST profesional (FastAPI)**
* Endpoint POST /predict
* Validaciones estrictas con Pydantic
* Manejo de errores 400, 503, 500
* Cargar modelo en el arranque, no entrenar en endpoints
#### **Documentación y buenas prácticas**
* README completo
* PEP8, tipado, docstrings
* Arquitectura por capas (API, Service, Pipeline, Training)
#### **Entregables**
* Código funcional en la estructura indicada.
* README con instrucciones claras.
* Guía de pruebas de API.
* Evidencia de entrenamiento y métricas.

#### **Arquitectura del proyecto**

```
project/
│
├── app/
│   │
│   ├── main.py 
│   │
│   ├── api/
│   │   └── routes.py
│   ├── service/
│   │   └── model_service.py
│
├── data/
│
├── pipeline/
│   └── pipeline.py
│
├── training/
│   └── train.py
│
├── model/
│   └── pipeline.joblib
│
├── requirements.txt
└── README.md
```


`pipeline/pipeline.py`: datos, columnas, caché, y construcción del pipeline (base conceptual de ML).

In [ ]:
"""Funciones para cargar datos y construir el pipeline de ML."""
# Docstring de módulo: describe el propósito general del archivo.
# En este caso, define funciones relacionadas con:
# - Descarga de datos
# - Carga del dataset
# - Construcción de un pipeline de Machine Learning

from __future__ import annotations
# Permite usar anotaciones de tipos como strings (útil para forward references)
# Mejora compatibilidad y claridad en proyectos grandes

from pathlib import Path
# Proporciona una forma moderna y segura de manejar rutas de archivos

from typing import List, Tuple
# List: para listas tipadas (ej. List[str])
# Tuple: para retornos múltiples tipados (ej. X, y)

from urllib.request import urlretrieve
# Función estándar para descargar archivos desde una URL

import pandas as pd
# Librería principal para manejo de datos tabulares (DataFrames)

from sklearn.compose import ColumnTransformer
# Permite aplicar transformaciones distintas a diferentes columnas

from sklearn.impute import SimpleImputer
# Maneja valores faltantes (NaN) en los datos

from sklearn.linear_model import ElasticNet
# Modelo de regresión regularizada (L1 + L2)

from sklearn.pipeline import Pipeline
# Encadena pasos de preprocesamiento y modelado de forma reproducible

from sklearn.preprocessing import StandardScaler
# Escala los datos para que tengan media 0 y desviación estándar 1


def get_dataset_url() -> str:
    """Retorna la URL pública del dataset real."""
    # Esta función centraliza la URL del dataset
    # Facilita cambios futuros y mejora mantenibilidad
    return (
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"
        "00374/energydata_complete.csv"
    )


def get_cache_path() -> Path:
    """Retorna la ruta local de caché del dataset."""
    # Define dónde se almacenará el dataset descargado
    # Usar Path evita errores por separadores de sistema operativo
    return Path("data") / "energydata_complete.csv"


def get_feature_columns() -> List[str]:
    """Retorna las columnas explicativas usadas para el modelo."""
    # Lista explícita de variables independientes (X)
    # Permite control total sobre qué datos entran al modelo
    return ["T1", "RH_1", "T_out", "Windspeed"]


def get_target_column() -> str:
    """Retorna la columna objetivo del problema."""
    # Define la variable dependiente (y)
    # En este dataset representa el consumo energético
    return "Appliances"


def download_dataset(url: str, cache_path: Path) -> Path:
    """Descarga el dataset desde una URL pública si no existe en caché."""
    # Crea el directorio padre si no existe
    # parents=True crea carpetas intermedias
    # exist_ok=True evita errores si ya existen
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    # Verifica si el archivo ya fue descargado
    if cache_path.exists():
        # Si existe, no se vuelve a descargar
        return cache_path

    # Descarga el archivo desde la URL y lo guarda en disco
    urlretrieve(url, cache_path.as_posix())

    # Retorna la ruta del archivo descargado
    return cache_path


def load_dataset() -> pd.DataFrame:
    """Carga el dataset desde caché, descargándolo si es necesario."""
    # Obtiene la ruta definitiva del dataset
    dataset_path = download_dataset(
        get_dataset_url(),
        get_cache_path()
    )

    # Lee el archivo CSV y lo convierte en un DataFrame
    df = pd.read_csv(dataset_path)

    # Define las columnas que se usarán (features + target)
    columns = get_feature_columns() + [get_target_column()]

    # Retorna una copia del DataFrame con solo las columnas necesarias
    # .copy() evita efectos colaterales en memoria
    return df[columns].copy()


def build_pipeline(feature_columns: List[str]) -> Pipeline:
    """Construye el pipeline de preprocesamiento y modelo."""
    # Pipeline exclusivo para variables numéricas
    numeric_transformer = Pipeline(
        steps=[
            # Paso 1: imputación de valores faltantes
            # Usa la mediana por ser robusta ante outliers
            ("imputer", SimpleImputer(strategy="median")),

            # Paso 2: escalado estándar
            # Fundamental para modelos regularizados como ElasticNet
            ("scaler", StandardScaler()),
        ]
    )

    # ColumnTransformer aplica el pipeline solo a las columnas indicadas
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",                   # Nombre del transformador
                numeric_transformer,     # Pipeline a aplicar
                feature_columns          # Columnas numéricas
            )
        ]
    )

    # Modelo de regresión ElasticNet
    # alpha controla la fuerza de la regularización
    # l1_ratio balancea L1 (Lasso) y L2 (Ridge)
    model = ElasticNet(
        alpha=0.1,
        l1_ratio=0.5,
        random_state=42
    )

    # Pipeline final: preprocesamiento + modelo
    # Garantiza consistencia entre entrenamiento y predicción
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", model)
        ]
    )


def split_features_target(
    df: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.Series]:
    """Separa variables explicativas y objetivo."""
    # Obtiene los nombres de las columnas explicativas
    feature_columns = get_feature_columns()

    # Obtiene el nombre de la variable objetivo
    target_column = get_target_column()

    # Retorna X (features) y y (target)
    return df[feature_columns], df[target_column]


`training/train.py:` entrenamiento, métricas y guardado del modelo (flujo ML completo).

In [ ]:
"""Entrenamiento y evaluación del modelo de consumo energético."""
# Docstring de módulo:
# Describe el propósito general del archivo.
# Este script se encarga de:
# - Entrenar el pipeline de ML
# - Evaluar el modelo con métricas estándar
# - Guardar el modelo entrenado para producción

from __future__ import annotations
# Permite usar anotaciones de tipos como strings
# Mejora compatibilidad y claridad en proyectos grandes

from math import sqrt
# Importa la función raíz cuadrada
# Se utiliza para calcular RMSE de forma manual si es necesario

from pathlib import Path
# Manejo moderno y seguro de rutas de archivos

import joblib
# Librería recomendada para serializar modelos de scikit-learn
# Más eficiente que pickle para objetos grandes

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
# Métricas estándar para evaluar modelos de regresión:
# - MAE: error promedio absoluto
# - MSE: error promedio cuadrático
# - R2: calidad explicativa del modelo

from sklearn.model_selection import train_test_split
# Divide los datos en conjuntos de entrenamiento y prueba
# Evita evaluar el modelo con datos vistos durante el entrenamiento

from pipeline.pipeline import (
    build_pipeline,
    get_feature_columns,
    load_dataset,
    split_features_target,
)
# Importa utilidades del módulo de pipeline:
# - build_pipeline: construye el pipeline completo
# - get_feature_columns: define variables explicativas
# - load_dataset: carga el dataset
# - split_features_target: separa X e y


def train_and_evaluate(model_path: Path) -> None:
    """Entrena el pipeline, evalúa y guarda el modelo entrenado."""
    # Función principal del entrenamiento:
    # - Entrena el modelo
    # - Evalúa su desempeño
    # - Guarda el pipeline entrenado en disco

    df = load_dataset()
    # Carga el dataset desde caché o desde internet
    # Retorna un DataFrame limpio y controlado

    features, target = split_features_target(df)
    # Separa el DataFrame en:
    # - features (X): variables independientes
    # - target (y): variable objetivo

    x_train, x_test, y_train, y_test = train_test_split(
        features,
        target,
        test_size=0.2,
        random_state=42,
    )
    # Divide los datos en:
    # - 80% entrenamiento
    # - 20% prueba
    # random_state asegura resultados reproducibles

    pipeline = build_pipeline(get_feature_columns())
    # Construye el pipeline completo:
    # - Preprocesamiento
    # - Modelo ElasticNet
    # Garantiza consistencia entre entrenamiento y predicción

    pipeline.fit(x_train, y_train)
    # Entrena el pipeline completo
    # Internamente:
    # - Aprende parámetros de imputación
    # - Aprende parámetros de escalado
    # - Ajusta el modelo de regresión

    predictions = pipeline.predict(x_test)
    # Genera predicciones sobre el conjunto de prueba
    # Simula un escenario real de datos no vistos

    mae = mean_absolute_error(y_test, predictions)
    # Calcula el error absoluto medio
    # Indica cuánto se equivoca el modelo en promedio

    mse = mean_squared_error(y_test, predictions)
    # Calcula el error cuadrático medio
    # Penaliza más los errores grandes

    rmse = _compute_rmse(y_test, predictions, mse)
    # Calcula la raíz del MSE
    # Métrica más interpretable en unidades originales

    r2 = r2_score(y_test, predictions)
    # Mide qué proporción de la variabilidad explica el modelo
    # Valores cercanos a 1 indican buen ajuste

    print("Métricas de evaluación en conjunto de prueba (test):")
    # Mensaje descriptivo para salida por consola

    print(f"- MAE  (Error absoluto medio)         : {mae:.4f}  (menor es mejor)")
    # Muestra MAE con formato decimal

    print(f"- MSE  (Error cuadrático medio)       : {mse:.4f}  (menor es mejor)")
    # Muestra MSE con formato decimal

    print(f"- RMSE (Raíz del MSE)                 : {rmse:.4f}  (menor es mejor)")
    # Muestra RMSE

    print(
        f"- R2   (Coeficiente de determinación): {r2:.4f}  (más cerca a 1 es mejor)"
    )
    # Muestra R² con interpretación directa

    model_path.parent.mkdir(parents=True, exist_ok=True)
    # Crea el directorio donde se guardará el modelo si no existe

    joblib.dump(pipeline, model_path)
    # Serializa y guarda el pipeline completo:
    # - Preprocesamiento
    # - Modelo entrenado

    print(f"Pipeline guardado en: {model_path}")
    # Mensaje de confirmación


def _compute_rmse(
    y_true,
    y_pred,
    mse_value: float,
) -> float:
    """Calcula RMSE con compatibilidad entre versiones de scikit-learn."""
    # Función auxiliar (helper)
    # Se define con "_" inicial para indicar uso interno

    try:
        # Intenta usar la métrica moderna de scikit-learn
        from sklearn.metrics import root_mean_squared_error

        return float(root_mean_squared_error(y_true, y_pred))
        # Retorna RMSE directamente

    except Exception:
        # Si la versión de sklearn no soporta esa métrica
        # se calcula manualmente

        return float(sqrt(mse_value))
        # RMSE = sqrt(MSE)


def main() -> None:
    """Punto de entrada del entrenamiento."""
    # Función principal del script
    # Facilita pruebas, reutilización y ejecución controlada

    model_path = Path("model") / "pipeline.joblib"
    # Define la ruta donde se guardará el modelo entrenado

    train_and_evaluate(model_path)
    # Ejecuta el flujo completo de entrenamiento


if __name__ == "__main__":
    # Este bloque se ejecuta solo si el archivo se corre directamente
    # No se ejecuta si se importa como módulo

    main()
    # Llama a la función principal


`app/service/model_service.py:` carga del modelo y lógica de inferencia (capa de servicio).

In [ ]:
"""Capa de servicio para el modelo de predicción."""
# Docstring del módulo:
# Este archivo implementa la CAPA DE SERVICIO del sistema.
# Su función es:
# - Encapsular el acceso al modelo entrenado
# - Aislar la API del detalle de ML
# - Proveer una interfaz clara y controlada para predicciones

from __future__ import annotations
# Permite usar anotaciones de tipos como strings
# Mejora compatibilidad y evita problemas de referencias circulares

from dataclasses import dataclass
# @dataclass reduce código repetitivo:
# - Genera automáticamente __init__, __repr__, __eq__, etc.

from pathlib import Path
# Manejo seguro y multiplataforma de rutas de archivos

from typing import Dict
# Tipado del diccionario de entrada para las features

import joblib
# Librería estándar para serializar/deserializar modelos de scikit-learn

import pandas as pd
# Se usa para construir DataFrames para inferencia

from sklearn.pipeline import Pipeline
# Tipo explícito del pipeline entrenado (preprocesamiento + modelo)

from pipeline.pipeline import get_feature_columns
# Importa la función que define el orden oficial de las features
# Evita errores críticos de columnas desordenadas

# ---------------------------------------------------------
# EXCEPCIÓN PERSONALIZADA
# ---------------------------------------------------------

class ModelNotLoadedError(RuntimeError):
    """
    Excepción de dominio del negocio.
    Se lanza cuando el modelo no está disponible.
    
    Beneficio:
    - Permite a la API mapear este error a HTTP 503
    - Evita usar excepciones genéricas poco claras
    """
    pass

# ---------------------------------------------------------
# SERVICIO DE MODELO
# ---------------------------------------------------------

@dataclass
class ModelService:
    """
    Servicio de inferencia que encapsula el pipeline entrenado.

    Responsabilidades:
    - Cargar el modelo desde disco
    - Validar disponibilidad
    - Ejecutar predicciones de forma segura
    """

    model_path: Path
    # Ruta física donde se encuentra el modelo serializado

    pipeline: Pipeline
    # Pipeline entrenado:
    # Incluye preprocesamiento + modelo ML

    # -----------------------------------------------------
    # MÉTODO DE CARGA (FACTORY METHOD)
    # -----------------------------------------------------

    @classmethod
    def load(cls, model_path: Path) -> "ModelService":
        """
        Carga el pipeline desde disco y construye el servicio.

        Se implementa como método de clase para:
        - Controlar la creación del objeto
        - Garantizar que el servicio siempre tenga un modelo válido
        """
        if not model_path.exists():
            # Valida que el archivo del modelo exista
            raise ModelNotLoadedError("Modelo no disponible.")

        try:
            # Intenta cargar el pipeline serializado
            pipeline = joblib.load(model_path)
        except Exception as exc:  # pragma: no cover
            # Protección defensiva:
            # - Archivo corrupto
            # - Versión incompatible
            # - Error de lectura
            raise ModelNotLoadedError("Modelo no disponible.") from exc

        # Retorna una instancia completamente válida del servicio
        return cls(
            model_path=model_path,
            pipeline=pipeline,
        )

    # -----------------------------------------------------
    # MÉTODO DE PREDICCIÓN
    # -----------------------------------------------------

    def predict(self, features: Dict[str, float]) -> float:
        """
        Genera una predicción única a partir de las features.

        Parameters:
        - features: diccionario {nombre_feature: valor}

        Returns:
        - Predicción escalar (float)
        """

        # Obtiene el orden oficial de columnas esperado por el modelo
        feature_columns = get_feature_columns()

        # Convierte el diccionario en un DataFrame 1xN
        # IMPORTANTE:
        # - El orden de columnas debe coincidir exactamente
        # - Evita errores silenciosos en producción
        data = pd.DataFrame(
            [features],
            columns=feature_columns,
        )

        # Ejecuta la inferencia usando el pipeline completo
        # [0] porque predict devuelve un array
        prediction = self.pipeline.predict(data)[0]

        # Convierte a float nativo para evitar tipos numpy
        return float(prediction)


`app/api/routes.py:` contrato REST, validaciones y endpoint /predict (capa API).

In [ ]:
"""Rutas de la API para predicción."""
# Docstring del módulo:
# Este archivo define las rutas (endpoints) REST relacionadas
# con la predicción del modelo de Machine Learning.
# Aquí NO se entrena el modelo, solo se consume.

from __future__ import annotations
# Permite usar anotaciones de tipos como strings
# Evita problemas de referencias circulares y mejora compatibilidad

from typing import Any
# Tipo genérico usado para el retorno del endpoint

from fastapi import APIRouter, Request, status
# APIRouter: permite organizar endpoints por módulos
# Request: acceso al contexto de la aplicación
# status: constantes de códigos HTTP

from pydantic import BaseModel, Field, ConfigDict
# BaseModel: define contratos de entrada/salida
# Field: validaciones y metadata de campos
# ConfigDict: configuración avanzada de validación (Pydantic v2)

from app.service.model_service import (
    ModelNotLoadedError,
    ModelService,
)
# Importa el servicio de inferencia y la excepción de dominio
# La API NO conoce detalles del modelo, solo del servicio

# ---------------------------------------------------------
# CONFIGURACIÓN DEL ROUTER
# ---------------------------------------------------------

router = APIRouter()
# Crea un router independiente que luego se registra en la app principal
# Facilita escalabilidad y organización del proyecto

# ---------------------------------------------------------
# MODELO DE ENTRADA (REQUEST DTO)
# ---------------------------------------------------------

class PredictionRequest(BaseModel):
    """
    Esquema de entrada para predicciones.
    Define exactamente qué datos acepta la API.
    """

    model_config = ConfigDict(
        extra="forbid",
        # Rechaza campos adicionales no definidos
        # Previene:
        # - Errores silenciosos
        # - Ataques por inyección de payloads inesperados

        json_schema_extra={
            # Información adicional para la documentación Swagger
            "examples": [
                {
                    "T1": 21.0,
                    "RH_1": 45.0,
                    "T_out": 10.0,
                    "Windspeed": 3.5,
                }
            ]
        },
    )

    # -------------------------
    # VARIABLES DE ENTRADA
    # -------------------------

    T1: float = Field(
        ...,
        ge=0.0,
        le=50.0,
        description="Temperatura en cocina (C)",
    )
    # Temperatura interior
    # Validación física razonable

    RH_1: float = Field(
        ...,
        ge=0.0,
        le=100.0,
        description="Humedad en cocina (%)",
    )
    # Humedad relativa (%)
    # Evita valores imposibles

    T_out: float = Field(
        ...,
        ge=-30.0,
        le=50.0,
        description="Temperatura exterior (C)",
    )
    # Temperatura exterior
    # Rango realista para datos ambientales

    Windspeed: float = Field(
        ...,
        ge=0.0,
        le=20.0,
        description="Velocidad del viento (m/s)",
    )
    # Velocidad del viento
    # Previene valores no físicos

# ---------------------------------------------------------
# MODELO DE SALIDA (RESPONSE DTO)
# ---------------------------------------------------------

class PredictionResponse(BaseModel):
    """
    Esquema de respuesta de predicción.
    Define el contrato de salida de la API.
    """

    model_config = ConfigDict(
        json_schema_extra={
            "examples": [
                {
                    "prediction": 120.45,
                    "model": "ElasticNet v1.0",
                    "unit": "energy consumption (Wh)",
                }
            ]
        }
    )

    prediction: float
    # Valor numérico estimado por el modelo

    model: str
    # Identificador del modelo usado
    # Útil para versionado y trazabilidad

    unit: str
    # Unidad de la predicción
    # Evita ambigüedad en clientes externos

# ---------------------------------------------------------
# FUNCIÓN DE ACCESO AL SERVICIO
# ---------------------------------------------------------

def get_service(request: Request) -> ModelService:
    """
    Recupera el servicio del modelo desde el estado de la app.
    Centraliza la validación de disponibilidad del modelo.
    """
    service = request.app.state.model_service
    # Accede al servicio cargado durante el startup

    if service is None:
        # Si el modelo no está disponible
        raise ModelNotLoadedError("Modelo no cargado.")
        # Esta excepción será traducida a HTTP 503

    return service
    # Devuelve un servicio listo para inferencia

# ---------------------------------------------------------
# ENDPOINT DE PREDICCIÓN
# ---------------------------------------------------------

@router.post(
    "/predict",
    response_model=PredictionResponse,
    status_code=status.HTTP_200_OK,
    summary="Predice consumo energético",
    response_description="Predicción del consumo en Wh",
    tags=["predictions"],
)
def predict(
    request: PredictionRequest,
    http_request: Request,
) -> Any:
    """
    Endpoint REST para predecir el consumo energético.

    Flujo:
    1. Validación automática del payload (Pydantic)
    2. Obtención del servicio de modelo
    3. Ejecución de la inferencia
    4. Construcción de la respuesta
    """

    service = get_service(http_request)
    # Obtiene el servicio de inferencia (capa de negocio)

    payload = request.model_dump()
    # Convierte el modelo Pydantic a dict plano
    # Garantiza que los datos ya fueron validados

    prediction = service.predict(payload)
    # Ejecuta la predicción usando el pipeline entrenado

    return PredictionResponse(
        prediction=prediction,
        model="ElasticNet v1.0",
        unit="energy consumption (Wh)",
    )
    # Devuelve respuesta estructurada y documentada


`app/main.py:` creación de la app, manejo de errores y arranque de Uvicorn (bootstrap del backend).

In [ ]:
"""Aplicación FastAPI para predicción de consumo energético."""
# Docstring del módulo:
# Este archivo define el punto de entrada principal de la aplicación FastAPI.
# Aquí se configuran:
# - El ciclo de vida (startup / shutdown)
# - El manejo global de errores
# - El servidor ASGI (Uvicorn)
# - La carga del modelo entrenado

from __future__ import annotations
# Permite usar anotaciones de tipos futuras como strings
# Evita problemas de dependencias circulares entre módulos

import logging
# Librería estándar para registrar eventos (logs)
# Fundamental en aplicaciones productivas

import sys
# Permite acceder y modificar sys.path
# Se usa para asegurar que el proyecto se importe correctamente

from contextlib import asynccontextmanager
# Utilidad moderna para manejar el ciclo de vida de FastAPI
# Reemplaza eventos on_startup / on_shutdown

from pathlib import Path
# Manejo seguro y multiplataforma de rutas de archivos

import threading
# Permite ejecutar tareas en segundo plano (hilos)

import time
# Usado para introducir una pequeña espera controlada

import webbrowser
# Permite abrir automáticamente el navegador web

from fastapi import FastAPI, Request, status
# FastAPI: framework principal
# Request: acceso al contexto de la petición
# status: constantes de códigos HTTP

from fastapi.exceptions import RequestValidationError
# Excepción lanzada cuando falla la validación de Pydantic

from fastapi.responses import JSONResponse
# Permite devolver respuestas JSON personalizadas

import uvicorn
# Servidor ASGI recomendado para FastAPI

# ---------------------------------------------------------
# CONFIGURACIÓN DEL PATH DEL PROYECTO
# ---------------------------------------------------------

ROOT_DIR = Path(__file__).resolve().parents[1]
# Calcula la raíz del proyecto:
# __file__ → archivo actual
# parents[1] → sube dos niveles en la jerarquía

if str(ROOT_DIR) not in sys.path:
    # Verifica si la raíz ya está en el path de Python
    sys.path.insert(0, str(ROOT_DIR))
    # Inserta la raíz del proyecto para permitir imports absolutos

# ---------------------------------------------------------
# IMPORTS INTERNOS DEL PROYECTO
# ---------------------------------------------------------

from app.api.routes import router
# Importa el router donde están definidos los endpoints

from app.service.model_service import ModelNotLoadedError, ModelService
# Importa:
# - El servicio que encapsula el modelo
# - La excepción de dominio cuando el modelo no está disponible

# ---------------------------------------------------------
# CONFIGURACIÓN DE LOGGING
# ---------------------------------------------------------

logging.basicConfig(level=logging.INFO)
# Configura el nivel mínimo de logging
# INFO es adecuado para ambientes de desarrollo y staging

logger = logging.getLogger(__name__)
# Logger específico de este módulo
# Permite identificar de dónde proviene cada log

# ---------------------------------------------------------
# CICLO DE VIDA DE LA APLICACIÓN (LIFESPAN)
# ---------------------------------------------------------

@asynccontextmanager
# Decorador que define un contexto asíncrono
# Se ejecuta en startup y shutdown de la aplicación

async def lifespan(app: FastAPI):
    """
    Manejo de eventos de ciclo de vida de la aplicación.
    Aquí se cargan recursos críticos (modelo ML).
    """
    model_path = Path("model") / "pipeline.joblib"
    # Ruta donde se espera encontrar el modelo entrenado

    try:
        # Intenta cargar el modelo al iniciar la app
        app.state.model_service = ModelService.load(model_path)
        # Guarda el servicio en el estado global de la app
    except ModelNotLoadedError:
        # Si el modelo no existe o falla la carga
        app.state.model_service = None
        # Marca explícitamente que el modelo no está disponible
        logger.error("Modelo no disponible en startup.")
        # Registra el error para monitoreo

    yield
    # A partir de aquí la aplicación empieza a atender requests
    # (No se necesita código de shutdown en este caso)

# ---------------------------------------------------------
# FACTORY DE LA APLICACIÓN
# ---------------------------------------------------------

def create_app() -> FastAPI:
    """
    Crea y configura la aplicación FastAPI.
    Se usa el patrón Factory para facilitar testing y despliegue.
    """
    app = FastAPI(
        title="Energy Consumption Prediction API",
        # Nombre visible en Swagger

        version="1.0.0",
        # Versión de la API (importante para versionado)

        lifespan=lifespan,
        # Asocia el ciclo de vida definido arriba
    )

    app.include_router(router)
    # Registra las rutas definidas en app.api.routes

    # -----------------------------------------------------
    # MANEJADORES GLOBALES DE EXCEPCIONES
    # -----------------------------------------------------

    @app.exception_handler(RequestValidationError)
    # Captura errores de validación de entrada (Pydantic)
    def validation_exception_handler(
        _request: Request,
        exc: RequestValidationError,
    ) -> JSONResponse:
        return JSONResponse(
            status_code=status.HTTP_400_BAD_REQUEST,
            # Código 400: error del cliente
            content={
                "error": "Datos inválidos.",
                "details": exc.errors(),
            },
            # Devuelve detalles claros del error
        )

    @app.exception_handler(ModelNotLoadedError)
    # Maneja errores cuando el modelo no está disponible
    def model_not_loaded_handler(
        _request: Request,
        _exc: ModelNotLoadedError,
    ) -> JSONResponse:
        return JSONResponse(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            # 503: servicio temporalmente no disponible
            content={"error": "Modelo no cargado."},
        )

    @app.exception_handler(Exception)
    # Captura cualquier excepción no controlada
    def generic_exception_handler(
        _request: Request,
        exc: Exception,
    ) -> JSONResponse:
        logger.exception("Error interno no controlado: %s", exc)
        # Registra el stacktrace completo
        return JSONResponse(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            # 500: error interno del servidor
            content={"error": "Error interno."},
        )

    return app
    # Devuelve la aplicación completamente configurada

# ---------------------------------------------------------
# INSTANCIA GLOBAL DE LA APP
# ---------------------------------------------------------

app = create_app()
# FastAPI detecta automáticamente esta variable como la app ASGI

# ---------------------------------------------------------
# EJECUCIÓN LOCAL CON UVICORN
# ---------------------------------------------------------

def main() -> None:
    """
    Ejecuta la aplicación con Uvicorn para pruebas locales.
    """
    def _open_docs() -> None:
        # Función auxiliar que abre Swagger automáticamente
        time.sleep(1.5)
        # Espera a que el servidor esté arriba
        webbrowser.open_new_tab("http://localhost:8000/docs")
        # Abre la documentación interactiva

    threading.Thread(
        target=_open_docs,
        daemon=True,
    ).start()
    # Lanza el hilo sin bloquear el servidor

    uvicorn.run(
        app,
        host="127.0.0.1",
        # Solo accesible localmente
        port=8000,
        # Puerto estándar de desarrollo
        reload=False,
        # En producción se recomienda False
    )

# ---------------------------------------------------------
# PUNTO DE ENTRADA DEL SCRIPT
# ---------------------------------------------------------

if __name__ == "__main__":
    main()
    # Ejecuta la app solo si el archivo se corre directamente


### **Para ejecuTar la app, leer el archivo TESTING_GUIDE.PDF